# 07 · Analysis, figures & report  (Phase **P7**)

**This notebook needs NO GPU and NO dataset** — it only reads the small result files that P1–P6 saved
to Drive, then builds the master results table, runs significance tests, regenerates the publication
figures, and emits a report-ready §5–§8 markdown. Run it on a **CPU runtime** anytime.

It is **defensive**: any phase whose results aren't on Drive yet is simply skipped with a note, so you
can run it after P0–P5 and re-run once P6 finishes.


## 1 · Mount Drive + helpers (no repo/torch needed)

In [ ]:
import os, json
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/fedsat"
except Exception:
    DRIVE = "artifacts"
RESULTS = f"{DRIVE}/results"
OUTDIR = f"{RESULTS}/p7_analysis"; os.makedirs(OUTDIR, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
try:
    import seaborn as sns
except Exception:
    import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seaborn"]); import seaborn as sns
from scipy import stats

def read_json(p):
    try:
        with open(p) as f: return json.load(f)
    except Exception: return None

def read_jsonl(p):
    try:
        return [json.loads(l) for l in open(p) if l.strip()]
    except Exception: return []

def savefig(name):
    plt.savefig(os.path.join(OUTDIR, name), dpi=150, bbox_inches="tight")
    print("saved figure:", name)

def ci95(vals):
    v = np.asarray(vals, dtype=float); n = len(v)
    if n == 0: return (float("nan"), float("nan"), float("nan"))
    m = v.mean()
    if n == 1: return (m, float("nan"), float("nan"))
    sem = v.std(ddof=1) / np.sqrt(n)
    h = stats.t.ppf(0.975, n - 1) * sem
    return (m, m - h, m + h)

print("reading results from", RESULTS)


## 2 · Which phase results are present?

In [ ]:
PATHS = {
    "P1_centralized": f"{RESULTS}/p1_centralized/metrics.json",
    "P2_fedavg_iid":  f"{RESULTS}/p2_fedavg_iid/metrics.json",
    "P3_noniid":      f"{RESULTS}/p3_noniid_sweep/results.jsonl",
    "P4_pftl":        f"{RESULTS}/p4_pftl/results.jsonl",
    "P5_scale_comm":  f"{RESULTS}/p5_scale_comm/results.jsonl",
    "P6_loco":        f"{RESULTS}/p6_loco/results.jsonl",
}
present = {}
for k, p in PATHS.items():
    ok = os.path.exists(p)
    present[k] = ok
    print(f"{'FOUND  ' if ok else 'MISSING'}  {k:16}  {p}")


## 3 · Master results table (headline numbers per phase)

In [ ]:
rows = []

# --- P1 centralized ---
p1 = read_json(PATHS["P1_centralized"])
if p1:
    m = p1.get("metrics", p1)
    rows.append(["P1", "Centralized (upper bound)", "global-test acc",
                 round(float(m.get("accuracy", float("nan"))), 4)])
    if "macro_f1" in m:
        rows.append(["P1", "Centralized", "macro-F1", round(float(m["macro_f1"]), 4)])

# --- P2 FedAvg on IID (Gate G4) ---
p2 = read_json(PATHS["P2_fedavg_iid"])
if p2:
    s = p2.get("summary", {})
    acc = s.get("test_accuracy_at_best", s.get("best_accuracy", p2.get("best_metrics", {}).get("accuracy")))
    if acc is not None:
        rows.append(["P2", "FedAvg IID (G4 check)", "global-test acc", round(float(acc), 4)])

# --- P3 non-IID sweep: FedAvg/FedProx at extreme alphas ---
p3 = read_jsonl(PATHS["P3_noniid"])
if p3:
    d3 = pd.DataFrame(p3)
    for reg in ["fedavg", "fedprox", "local_only"]:
        sub = d3[d3.get("regime") == reg] if "regime" in d3 else d3.iloc[0:0]
        for a in sorted(set(d3["alpha"].dropna())) if "alpha" in d3 else []:
            ss = sub[sub["alpha"] == a]
            if len(ss):
                rows.append(["P3", f"{reg} (alpha={a})", "global-test acc",
                             round(float(ss["global_test_accuracy"].mean()), 4)])

# --- P4 proposed method (shift ON): mean per-client test, mean +/- 95% CI over seeds ---
p4 = read_jsonl(PATHS["P4_pftl"])
if p4:
    d4 = pd.DataFrame(p4)
    on = d4[d4["shift"] == True]
    for method in ["fedavg_bn", "fedprox_bn", "fedbn", "fedbn_prox", "fedavg_gn"]:
        v = on[on["method"] == method]["mean_test_acc"].values
        if len(v):
            m, lo, hi = ci95(v)
            ci = f" (95% CI {lo:.3f}-{hi:.3f})" if lo == lo else ""
            rows.append(["P4", f"{method} (shift ON)", "mean per-client acc", f"{m:.4f}{ci}"])

# --- P5 scale + compression headlines ---
p5 = read_jsonl(PATHS["P5_scale_comm"])
if p5:
    d5 = pd.DataFrame(p5)
    for K in sorted(set(d5["K"])):
        ss = d5[(d5["K"] == K) & (d5["fraction_fit"] == 1.0) & (d5["compress"] == "none")]
        if len(ss):
            rows.append(["P5", f"scale K={int(K)}", "global-test acc", round(float(ss["test_acc"].mean()), 4)])
    comp = d5[(d5["fraction_fit"] == 1.0)]
    for cname in ["none", "8bit", "topk@0.1", "topk@0.01"]:
        ss = comp[comp["compress"] == cname]
        if len(ss):
            rows.append(["P5", f"compression {cname}", "global-test acc", round(float(ss["test_acc"].mean()), 4)])

# --- P6 LOCO: FedAvg on unseen region, base vs +AdaBN ---
p6 = read_jsonl(PATHS["P6_loco"])
if p6:
    d6 = pd.DataFrame(p6)
    fa = d6[d6["method"] == "fedavg"]
    if len(fa):
        rows.append(["P6", "FedAvg on UNSEEN region", "acc (base)", round(float(fa["base_acc"].mean()), 4)])
        if "adabn_acc" in fa and fa["adabn_acc"].notna().any():
            rows.append(["P6", "FedAvg on UNSEEN region", "acc (+AdaBN)", round(float(fa["adabn_acc"].mean()), 4)])

master = pd.DataFrame(rows, columns=["phase", "setting", "metric", "value"])
print(master.to_string(index=False))
master.to_csv(os.path.join(OUTDIR, "master_results.csv"), index=False)
print("\nsaved -> master_results.csv")


## 4 · Significance of the proposed method (P4, shift ON)

Same seeds across methods ⇒ use a **paired** comparison. We report the per-seed differences, the
mean difference with a 95% CI, and a paired t-test. (With 3 seeds a p-value is weak; the all-positive
per-seed differences are the stronger evidence.)


In [ ]:
if p4:
    d4 = pd.DataFrame(p4)
    on = d4[d4["shift"] == True].pivot_table(index="seed", columns="method", values="mean_test_acc")
    print("per-seed mean per-client accuracy (shift ON):"); print(on.round(4).to_string()); print()
    if "fedbn" in on:
        for base in ["fedavg_bn", "fedprox_bn"]:
            if base in on:
                pair = on[["fedbn", base]].dropna()
                diff = (pair["fedbn"] - pair[base]).values
                m, lo, hi = ci95(diff)
                line = f"FedBN - {base}: per-seed {np.round(diff,4).tolist()} | mean {m:+.4f}"
                if lo == lo:
                    line += f" (95% CI {lo:+.4f}..{hi:+.4f})"
                    if len(diff) >= 2:
                        t, p = stats.ttest_rel(pair["fedbn"], pair[base])
                        line += f" | paired t p={p:.3f}"
                print(line)
    print("\nInterpretation: positive & CI-separated => FedBN significantly beats the baseline under feature shift.")
else:
    print("P4 results not found yet.")


## 5 · Figure — P3 heterogeneity curve (accuracy vs alpha)

In [ ]:
if p3:
    d3 = pd.DataFrame(p3)
    cent = d3[d3["regime"] == "centralized"]["global_test_accuracy"].mean() if "regime" in d3 else float("nan")
    fed = d3[d3["regime"].isin(["local_only", "fedavg", "fedprox"])]
    agg = fed.groupby(["regime", "alpha"])["global_test_accuracy"].mean().reset_index()
    alphas = sorted(set(fed["alpha"]))
    xs = range(len(alphas))
    plt.figure(figsize=(8, 5))
    for reg, color in [("fedprox", "#229954"), ("fedavg", "#d35400"), ("local_only", "#8e44ad")]:
        s = agg[agg["regime"] == reg].set_index("alpha").reindex(alphas)
        plt.plot(list(xs), s["global_test_accuracy"], "-o", label=reg, color=color)
    if cent == cent:
        plt.axhline(cent, ls="--", color="#444", label=f"centralized ({cent:.3f})")
    plt.xticks(list(xs), [str(a) for a in alphas])
    plt.xlabel("Dirichlet alpha (left=severe non-IID, right=IID)"); plt.ylabel("global-test accuracy")
    plt.ylim(0, 1); plt.legend(loc="lower right"); plt.grid(alpha=0.3)
    plt.title("P3: regimes vs data heterogeneity"); savefig("fig_p3_heterogeneity.png"); plt.show()
else:
    print("P3 results not found — skipping figure.")


## 6 · Figure — P4 proposed method (FedBN) under sensor shift

In [ ]:
if p4:
    d4 = pd.DataFrame(p4)
    agg = d4.groupby(["method", "shift"])["mean_test_acc"].agg(["mean", "std"]).reset_index()
    order = ["fedavg_bn", "fedprox_bn", "fedbn", "fedbn_prox", "fedavg_gn"]
    x = np.arange(len(order)); w = 0.38
    plt.figure(figsize=(9, 5))
    for i, (sh, color) in enumerate([(False, "#95a5a6"), (True, "#2980b9")]):
        means = [float(agg[(agg["method"] == m) & (agg["shift"] == sh)]["mean"].iloc[0])
                 if len(agg[(agg["method"] == m) & (agg["shift"] == sh)]) else np.nan for m in order]
        errs = [float(agg[(agg["method"] == m) & (agg["shift"] == sh)]["std"].fillna(0).iloc[0])
                if len(agg[(agg["method"] == m) & (agg["shift"] == sh)]) else 0.0 for m in order]
        plt.bar(x + (i - 0.5) * w, means, w, yerr=errs, capsize=4,
                label=("sensor shift ON" if sh else "shift OFF"), color=color)
    plt.xticks(x, order, rotation=15, ha="right"); plt.ylabel("mean per-client test accuracy")
    plt.ylim(0, 1); plt.legend(); plt.grid(axis="y", alpha=0.3)
    plt.title("P4: proposed FedBN vs baselines (mean +/- std over seeds)")
    savefig("fig_p4_fedbn.png"); plt.show()
else:
    print("P4 results not found — skipping figure.")


## 7 · Figure — P5 scale, participation & compression

In [ ]:
if p5:
    d5 = pd.DataFrame(p5)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

    sc = (d5[(d5["fraction_fit"] == 1.0) & (d5["compress"] == "none")]
          .groupby("K")[["test_acc", "total_comm_mb"]].mean().sort_index())
    ax[0].plot(sc.index.astype(str), sc["test_acc"], "-o", color="#2980b9")
    ax[0].set_xlabel("clients K"); ax[0].set_ylabel("global-test acc"); ax[0].set_ylim(0, 1)
    ax[0].set_title("Scale"); ax[0].grid(alpha=0.3)

    K0 = int(d5["K"].mode().iloc[0]) if "K" in d5 else 20
    pp = (d5[(d5["K"] == 20) & (d5["compress"] == "none")]
          .groupby("fraction_fit")[["test_acc"]].mean().sort_index())
    if len(pp):
        ax[1].plot([str(f) for f in pp.index], pp["test_acc"], "-o", color="#16a085")
    ax[1].set_xlabel("fraction_fit (K=20)"); ax[1].set_ylabel("global-test acc"); ax[1].set_ylim(0, 1)
    ax[1].set_title("Partial participation"); ax[1].grid(alpha=0.3)

    comp = (d5[(d5["fraction_fit"] == 1.0)].groupby("compress")[["test_acc", "total_comm_mb"]].mean())
    if "none" in comp.index:
        downlink = comp.loc["none", "total_comm_mb"] / 2.0
        comp["uplink_mb"] = comp["total_comm_mb"] - downlink
        for name in ["none", "8bit", "topk@0.1", "topk@0.01"]:
            if name in comp.index:
                ax[2].scatter(comp.loc[name, "uplink_mb"], comp.loc[name, "test_acc"], s=80)
                ax[2].annotate(name, (comp.loc[name, "uplink_mb"], comp.loc[name, "test_acc"]),
                               textcoords="offset points", xytext=(6, 4), fontsize=8)
        ax[2].set_xscale("log")
    ax[2].set_xlabel("uplink MB (log)"); ax[2].set_ylabel("global-test acc"); ax[2].set_ylim(0, 1)
    ax[2].set_title("Compression (uplink)"); ax[2].grid(alpha=0.3)
    plt.tight_layout(); savefig("fig_p5_scale_comm.png"); plt.show()
else:
    print("P5 results not found — skipping figure.")


## 8 · Figure — P6 leave-one-region-out generalization + AdaBN

In [ ]:
if p6:
    d6 = pd.DataFrame(p6)
    conds, means, errs = [], [], []
    for method in ["fedavg", "fedprox"]:
        sub = d6[d6["method"] == method]
        if len(sub):
            conds += [method, method + "+adabn"]
            means += [sub["base_acc"].mean(), sub["adabn_acc"].mean()]
            errs += [sub["base_acc"].std(ddof=1) if len(sub) > 1 else 0.0,
                     sub["adabn_acc"].std(ddof=1) if len(sub) > 1 else 0.0]
    x = np.arange(len(conds))
    plt.figure(figsize=(8, 5))
    plt.bar(x, means, yerr=np.nan_to_num(errs), capsize=4,
            color=["#d35400", "#e67e22", "#8e44ad", "#9b59b6"][:len(conds)])
    plt.xticks(x, conds, rotation=15, ha="right"); plt.ylabel("accuracy on UNSEEN region")
    plt.ylim(0, 1); plt.grid(axis="y", alpha=0.3)
    plt.title("P6: generalization to an unseen region (+AdaBN)")
    savefig("fig_p6_loco.png"); plt.show()
else:
    print("P6 results not found — skipping figure.")


## 9 · Figure — P1 centralized confusion matrix / per-class F1

In [ ]:
if p1 and "metrics" in p1 and "confusion_matrix" in p1["metrics"]:
    m = p1["metrics"]
    cm = np.array(m["confusion_matrix"]); names = list(m.get("per_class_f1", {}).keys()) or [str(i) for i in range(len(cm))]
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names)
    plt.xlabel("predicted"); plt.ylabel("true"); plt.title("P1 centralized — confusion (global test)")
    savefig("fig_p1_confusion.png"); plt.show()
    if "per_class_f1" in m:
        pcf = m["per_class_f1"]
        plt.figure(figsize=(10, 4)); plt.bar(list(pcf.keys()), list(pcf.values()), color="#229954")
        plt.ylim(0, 1); plt.xticks(rotation=45, ha="right"); plt.ylabel("F1"); plt.title("P1 per-class F1")
        savefig("fig_p1_perclass.png"); plt.show()
else:
    print("P1 metrics/confusion not found — skipping figure.")


## 10 · Emit report-ready §5–§8 markdown (real numbers)

In [ ]:
def g(rows_df, phase, setting, default="n/a"):
    r = rows_df[(rows_df["phase"] == phase) & (rows_df["setting"].str.contains(setting, regex=False))]
    return str(r["value"].iloc[0]) if len(r) else default

def to_md(df):
    try:
        return df.to_markdown(index=False)          # needs 'tabulate'
    except Exception:
        return "```\n" + df.to_string(index=False) + "\n```"

L = ["# Results & Discussion (auto-generated from executed runs)", ""]
L += ["_All numbers below are pulled directly from the P1-P6 result files on Drive; nothing hand-typed._", ""]
L += ["## Master results", "", to_md(master), ""]

L += ["## Key findings", ""]
if p1:
    L.append(f"- **Centralized upper bound:** {g(master,'P1','Centralized (upper')} global-test accuracy on real EuroSAT "
             "(all 10 classes, no class collapse) — the credible ceiling the prior project lacked.")
if p2:
    L.append(f"- **FL correctness (G4):** FedAvg on IID data reaches {g(master,'P2','FedAvg IID')}, matching centralized — "
             "the federated loop is correct (the prior project collapsed 0.99->0.33 here).")
if p3:
    L.append("- **Heterogeneity (P3):** FedAvg degrades as label skew increases; FedProx is more robust at severe skew; "
             "a lone client's global accuracy collapses while its in-domain accuracy stays high.")
if p4:
    d4 = pd.DataFrame(p4); on = d4[d4["shift"] == True]
    fb = ci95(on[on["method"] == "fedbn"]["mean_test_acc"].values)
    fa = ci95(on[on["method"] == "fedavg_bn"]["mean_test_acc"].values)
    L.append(f"- **Proposed method (P4):** under simulated sensor shift, **FedBN {fb[0]:.3f}** vs FedAvg {fa[0]:.3f} "
             "(per-client test accuracy) — a CI-separated improvement; FedBN's personalized BatchNorm absorbs feature shift. "
             "With no shift FedBN sits just below the baselines (the expected, disclosed trade-off).")
if p5:
    L.append(f"- **Scale & communication (P5):** accuracy holds from small to many clients (see table); 8-bit uplink "
             "compression is ~lossless while top-k sparsification trades accuracy for communication.")
if p6:
    L.append(f"- **Cross-domain generalization (P6):** on an UNSEEN region, FedAvg reaches {g(master,'P6','UNSEEN region')} base"
             f"; label-free AdaBN adaptation ({g(master,'P6','+AdaBN','')}) recovers accuracy — BatchNorm statistics are the "
             "key lever for feature shift, for both participating (FedBN) and unseen (AdaBN) regions.")

L += ["", "## Honest limitations", "",
      "- Cross-region/sensor heterogeneity is *simulated* (Dirichlet label skew + seeded photometric shifts), not real "
      "multi-continent data; it is a controlled, reproducible study of the mechanism.",
      "- EuroSAT has a high accuracy ceiling; the informative axes are heterogeneity, scale, and unseen-region "
      "generalization, not absolute IID accuracy.",
      "- Federated learning here provides data minimization, not a formal privacy guarantee (DP-SGD is future work)."]

report = "\n".join(L)
with open(os.path.join(OUTDIR, "report_results.md"), "w", encoding="utf-8") as f:
    f.write(report)
print("wrote report_results.md to", OUTDIR)
print("\n" + report[:1500] + "\n...")


## Done — P7 complete

Under `results/p7_analysis/` on Drive you now have: `master_results.csv`, `report_results.md`, and the
figures (`fig_p3_heterogeneity.png`, `fig_p4_fedbn.png`, `fig_p5_scale_comm.png`, `fig_p6_loco.png`,
`fig_p1_confusion.png`, …). Download those into the thesis for §5–§8.

Re-run this notebook any time (CPU is fine) — it always reflects whatever results are currently on
Drive, so once P6 finishes, re-running fills in the P6 rows/figure automatically.
